# EarthRisk AI Hackathon Brief

## Problem & Challenge

### Problem
- Insurance companies must daily evaluate risk for areas, properties and infrastructure.
- Climate change continuously alters the environment: temperature rise, vegetation loss, land use shifts, drought, and general area degradation can gradually change the risk profile of an asset or geographic zone.

### Challenge
- Access to satellite imagery and climate data exists, but the real difficulty lies in making sense of it.
- Being able to detect how an area changes over time, combine environmental change with temperature evolution, and convert this into clear, explainable risk insights that help insurers make better decisions.

## What The Solution Should Do
- Analyze historical and recent geospatial and satellite data alongside temperature data for a given area.
- Detect changes over time such as vegetation loss, land use change, increase in built-up surface, and changes in the thermal behavior of an area.
- Calculate a Climate / Insurance Risk Score based on temperature increase, heat exposure, vegetation loss, land stress, severity of change, and proximity to insured assets.
- Build a predictive model for which areas or assets are most likely to show increased insurance risk in the near future.
- Generate risk alerts, prioritization, and decision-support insights for areas needing increased attention, monitoring, or risk reassessment.


# Problem Statement

This repository is solving the exact narrative **heat + vegetation loss + urban expansion = rising insurance risk**.

That is verified by the implementation:
- the deterministic score has `heat_score`, `vegetation_loss_score`, and `urban_expansion_score` components,
- the feature store engineers vegetation loss, built-up gain, and heat-change features,
- the predictive layer estimates whether a hex cell's risk score will increase over the next 12 months,
- the alerts layer turns those outputs into explainable monitoring signals.

A higher risk score means a Phoenix-area H3 hex is showing stronger evidence of worsening heat stress, declining vegetation, increasing built-up pressure, land stress, and insured-asset exposure. This is an explainable geospatial early-warning system, not a claims model.


In [30]:
from __future__ import annotations

from pathlib import Path
import sys
import pickle

import geopandas as gpd
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config
from src.utils.paths import get_paths

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

config = load_config(PROJECT_ROOT / 'configs/phoenix.yaml')
paths = get_paths()

grid = gpd.read_parquet(paths.grid_path(config.hex_resolution))
landsat = pd.read_parquet(paths.landsat_features)
nlcd = pd.read_parquet(paths.nlcd_features)
dynamic_world = pd.read_parquet(paths.dynamic_world_features)
era5 = pd.read_parquet(paths.era5_features)
buildings = pd.read_parquet(paths.buildings_features)
feature_store = pd.read_parquet(paths.feature_store)
risk_scores = pd.read_parquet(paths.risk_scores)
training_table = pd.read_parquet(paths.training_table)
predictions = pd.read_parquet(paths.predictions_current)
alerts = pd.read_parquet(paths.alerts_current)
feature_importance = pd.read_csv(paths.feature_importance)

model_artifact = None
if paths.model_path.exists():
    with paths.model_path.open('rb') as handle:
        model_artifact = pickle.load(handle)

tables = {
    'grid': grid,
    'landsat': landsat,
    'nlcd': nlcd,
    'dynamic_world': dynamic_world,
    'era5': era5,
    'buildings': buildings,
    'feature_store': feature_store,
    'risk_scores': risk_scores,
    'training_table': training_table,
    'predictions': predictions,
    'alerts': alerts,
}

print(f'AOI: {config.aoi_name}')
print(f'Analysis years: {config.years.start}-{config.years.end}')
print(f'Hex resolution: {config.hex_resolution}')

AOI: greater_phoenix
Analysis years: 2018-2024
Hex resolution: 8


In [31]:
config_snapshot = pd.DataFrame(
    {
        'parameter': ['analysis_crs', 'hex_resolution', 'years.start', 'years.end', 'season_months', 'prediction_horizon_months', 'score_refresh_months', 'baseline_years', 'landsat_scale', 'nlcd_scale', 'dynamic_world_scale', 'era5_scale', 'risk_score_threshold', 'probability_threshold', 'min_building_count'],
        'value': [config.analysis_crs, config.hex_resolution, config.years.start, config.years.end, config.season_months, config.prediction_horizon_months, config.score_refresh_months, config.baseline_years, config.earth_engine.landsat_scale, config.earth_engine.nlcd_scale, config.earth_engine.dynamic_world_scale, config.earth_engine.era5_scale, config.alerts.risk_score_threshold, config.alerts.probability_threshold, config.alerts.min_building_count],
    }
)

inventory = pd.DataFrame([
    {'table': name, 'rows': len(table), 'columns': len(table.columns), 'years': sorted(table['year'].dropna().unique().tolist()) if 'year' in table.columns else None}
    for name, table in tables.items()
])

risk_weights = pd.DataFrame(
    {
        'component': ['heat_score', 'vegetation_loss_score', 'urban_expansion_score', 'land_stress_score', 'asset_exposure_score'],
        'weight': [0.35, 0.25, 0.20, 0.10, 0.10],
        'meaning': ['thermal stress and worsening heat', 'decline in vegetation cover', 'growth in built-up or converted land', 'supporting land degradation signal', 'insured-asset exposure proxy'],
    }
)

display(config_snapshot)
display(inventory)
display(risk_weights)
display(risk_scores[['risk_score', 'heat_score', 'vegetation_loss_score', 'urban_expansion_score', 'land_stress_score', 'asset_exposure_score']].describe().T)


,parameter,value
0,analysis_crs,EPSG:4326
1,hex_resolution,8
2,years.start,2018
3,years.end,2024
4,season_months,"[6, 7, 8, 9]"
5,prediction_horizon_months,12
6,score_refresh_months,6
7,baseline_years,"[2018, 2019, 2020]"
8,landsat_scale,30
9,nlcd_scale,30


,table,rows,columns,years
0,grid,3378,5,None
1,landsat,23646,9,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
2,nlcd,23646,11,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
3,dynamic_world,23646,12,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
4,era5,23646,9,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
5,buildings,3378,5,None
6,feature_store,23646,52,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
7,risk_scores,23646,8,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
8,training_table,20268,60,"[2018, 2019, 2020, 2021, 2022, 2023]"
9,predictions,3378,3,None


,component,weight,meaning
0,heat_score,0.35,thermal stress and worsening heat
1,vegetation_loss_score,0.25,decline in vegetation cover
2,urban_expansion_score,0.20,growth in built-up or converted land
3,land_stress_score,0.10,supporting land degradation signal
4,asset_exposure_score,0.10,insured-asset exposure proxy


,count,mean,std,min,25%,50%,75%,max
risk_score,23646.0,38.381251,9.235077,8.876817,31.730981,38.139979,45.020017,80.286785
heat_score,23646.0,48.592689,18.346711,10.900122,32.715960,49.016370,64.013574,96.359814
vegetation_loss_score,23646.0,38.956470,11.700984,12.500000,35.178404,37.500000,41.427960,87.500000
urban_expansion_score,23646.0,24.605434,13.702095,0.000000,16.526102,22.541233,28.662404,99.521319
land_stress_score,23646.0,24.227067,17.333475,0.000000,12.358720,20.732091,30.091251,98.258698
asset_exposure_score,23646.0,42.908984,19.807168,0.000000,27.224673,44.714253,59.300927,77.797826


In [32]:
EXCLUDED_COLUMNS = {'hex_id', 'year', 'prediction_year', 'risk_score_t_plus_1', 'risk_delta_12m', 'target_increase_12m'}
model_feature_columns = [c for c in training_table.columns if c not in EXCLUDED_COLUMNS and pd.api.types.is_numeric_dtype(training_table[c])]

def describe_feature(name: str) -> tuple[str, str]:
    if name.startswith('ndvi'):
        return 'Landsat', 'Vegetation greenness or its variability.'
    if name.startswith('ndbi'):
        return 'Landsat', 'Built-up surface proxy.'
    if name.startswith('bsi'):
        return 'Landsat', 'Bare-soil or disturbed-ground proxy.'
    if name.startswith('lst'):
        return 'Landsat', 'Land surface temperature metric.'
    if name.startswith('pct_') or name.startswith('delta_pct_'):
        return 'NLCD', 'Land-cover share or year-over-year change.'
    if name.startswith('dw_'):
        return 'Dynamic World', 'Class probability, land-cover share, or transition feature.'
    if name.startswith('air_temp') or name.startswith('hot_days') or name.startswith('extreme_hot_days') or name.startswith('max_consecutive_hot_days') or name.startswith('temp_trend'):
        return 'ERA5-Land', 'Air-temperature or heat-duration feature.'
    if name.startswith('building') or name.startswith('mean_building'):
        return 'Buildings', 'Asset exposure proxy from local building footprints.'
    if name in {'vegetation_index', 'built_index', 'barren_index', 'veg_loss_1y', 'veg_loss_3y', 'built_gain_1y', 'built_gain_3y', 'surface_heat_change_1y', 'surface_heat_change_3y', 'air_heat_change_1y', 'air_heat_change_3y'}:
        return 'Engineered', 'Cross-dataset change or composite index used for scoring and prediction.'
    if 'score' in name:
        return 'Score layer', 'Deterministic risk component or final score.'
    return 'Other', 'Additional numeric feature carried into modeling.'

feature_glossary = pd.DataFrame([
    {'feature': name, 'source_group': describe_feature(name)[0], 'meaning': describe_feature(name)[1]}
    for name in model_feature_columns
])

display(feature_glossary)


,feature,source_group,meaning
0,risk_score_t,Score layer,Deterministic risk component or final score.
1,ndvi_mean,Landsat,Vegetation greenness or its variability.
2,ndvi_std,Landsat,Vegetation greenness or its variability.
3,ndbi_mean,Landsat,Built-up surface proxy.
4,bsi_mean,Landsat,Bare-soil or disturbed-ground proxy.
5,lst_mean_summer,Landsat,Land surface temperature metric.
6,lst_p90_summer,Landsat,Land surface temperature metric.
7,valid_pixel_fraction,Other,Additional numeric feature carried into modeling.
8,pct_developed,NLCD,Land-cover share or year-over-year change.
9,pct_shrub_scrub,NLCD,Land-cover share or year-over-year change.


In [33]:
test_year = int(training_table['year'].max())
train_df = training_table[training_table['year'] < test_year].copy()
test_df = training_table[training_table['year'] == test_year].copy()

split_summary = pd.DataFrame(
    {
        'split': ['train', 'test'],
        'rows': [len(train_df), len(test_df)],
        'hex_cells': [train_df['hex_id'].nunique(), test_df['hex_id'].nunique()],
        'year_range': [f"{train_df['year'].min()}-{train_df['year'].max()}", f"{test_df['year'].min()}-{test_df['year'].max()}"],
        'positive_rate': [train_df['target_increase_12m'].mean(), test_df['target_increase_12m'].mean()],
        'mean_risk_score_t': [train_df['risk_score_t'].mean(), test_df['risk_score_t'].mean()],
        'mean_risk_delta_12m': [train_df['risk_delta_12m'].mean(), test_df['risk_delta_12m'].mean()],
    }
)

missingness = pd.DataFrame(
    {
        'feature': model_feature_columns,
        'train_missing_rate': [train_df[c].isna().mean() for c in model_feature_columns],
        'test_missing_rate': [test_df[c].isna().mean() for c in model_feature_columns],
    }
).sort_values(['test_missing_rate', 'train_missing_rate'], ascending=False)

display(split_summary)
display(missingness.head(20))
display(training_table.groupby('year')[['target_increase_12m', 'risk_delta_12m', 'risk_score_t']].mean().reset_index())


,split,rows,hex_cells,year_range,positive_rate,mean_risk_score_t,mean_risk_delta_12m
0,train,16890,3378,2018-2022,0.287448,36.996272,0.994817
1,test,3378,3378,2023-2023,0.289816,39.254944,5.177506


,feature,train_missing_rate,test_missing_rate
27,air_temp_mean_annual,0.280047,0.280047
28,air_temp_mean_summer,0.280047,0.280047
29,air_temp_anomaly_vs_baseline,0.280047,0.280047
30,hot_days_count,0.280047,0.280047
31,extreme_hot_days_count,0.280047,0.280047
32,max_consecutive_hot_days,0.280047,0.280047
0,risk_score_t,0.000000,0.000000
1,ndvi_mean,0.000000,0.000000
2,ndvi_std,0.000000,0.000000
3,ndbi_mean,0.000000,0.000000


,year,target_increase_12m,risk_delta_12m,risk_score_t
0,2018,0.123446,0.786210,34.280861
1,2019,0.467436,4.886099,35.067071
2,2020,0.065127,-8.411384,39.953170
3,2021,0.759325,12.596689,31.541785
4,2022,0.021906,-4.883531,44.138475
5,2023,0.289816,5.177506,39.254944


# Algorithm And Hyperparameters

The predictive layer is an **XGBoost binary classifier** trained on all numeric columns in the training table except the explicit exclusion list. Missing values are filled with feature medians. The model predicts the probability that a hex cell's risk score will materially increase over the next 12 months.

The production training code does not create a validation split, does not use early stopping, and does not calibrate probabilities.

# Holdout Evaluation Metrics

The production pipeline does not save evaluation metrics, so this notebook estimates them by re-training the same XGBoost setup on the analysis train split and scoring it on the temporal holdout test split.

These are analysis metrics rather than production artifacts, but they are the right way to answer how well the current model generalizes to a future year. The classification threshold below is `0.5`.

In [34]:
train_fill_values = train_df[model_feature_columns].median(numeric_only=True).fillna(0.0)
X_train = train_df[model_feature_columns].fillna(train_fill_values)
y_train = train_df['target_increase_12m'].astype(int)
X_test = test_df[model_feature_columns].fillna(train_fill_values)
y_test = test_df['target_increase_12m'].astype(int)

evaluation_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
)
evaluation_model.fit(X_train, y_train)

test_prob = evaluation_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

evaluation_metrics = pd.DataFrame(
    {
        'metric': ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision', 'log_loss', 'positive_rate_test', 'positive_rate_predicted'],
        'value': [
            accuracy_score(y_test, test_pred),
            precision_score(y_test, test_pred, zero_division=0),
            recall_score(y_test, test_pred, zero_division=0),
            f1_score(y_test, test_pred, zero_division=0),
            roc_auc_score(y_test, test_prob),
            average_precision_score(y_test, test_prob),
            log_loss(y_test, test_prob),
            y_test.mean(),
            test_pred.mean(),
        ],
    }
)

confusion = pd.DataFrame(
    confusion_matrix(y_test, test_pred),
    index=['actual_0', 'actual_1'],
    columns=['pred_0', 'pred_1'],
)

display(evaluation_metrics)
display(confusion)


,metric,value
0,accuracy,0.706631
1,precision,0.300000
2,recall,0.009193
3,f1,0.017839
4,roc_auc,0.692444
5,average_precision,0.427080
6,log_loss,0.875019
7,positive_rate_test,0.289816
8,positive_rate_predicted,0.008881


,pred_0,pred_1
actual_0,2378,21
actual_1,970,9


In [35]:
explicit_xgb_hyperparameters = pd.DataFrame(
    [
        ('n_estimators', 200),
        ('max_depth', 4),
        ('learning_rate', 0.05),
        ('subsample', 0.8),
        ('colsample_bytree', 0.8),
        ('objective', 'binary:logistic'),
        ('eval_metric', 'logloss'),
        ('random_state', 42),
        ('missing_value_fill', 'median per feature'),
        ('target_threshold', 'risk_delta_12m >= 8'),
        ('early_stopping', 'not used'),
        ('class_weighting', 'not set explicitly'),
        ('probability_calibration', 'not used')
    ],
    columns=['hyperparameter', 'value']
)

display(explicit_xgb_hyperparameters)
if model_artifact is not None:
    display(pd.Series(model_artifact['model'].get_params()).sort_index().to_frame('value'))

suffix_features = [c for c in model_feature_columns if c.endswith('_x') or c.endswith('_y')]
display(pd.DataFrame({'duplicated_suffix_feature': suffix_features}))
display(feature_importance)
display(pd.DataFrame({'metric': ['top_importance_feature', 'mean_predicted_probability', 'max_predicted_probability', 'alert_rows'], 'value': [feature_importance.iloc[0]['feature'] if not feature_importance.empty else None, predictions['prob_risk_increase_12m'].mean(), predictions['prob_risk_increase_12m'].max(), len(alerts)]}))


,hyperparameter,value
0,n_estimators,200
1,max_depth,4
2,learning_rate,0.05
3,subsample,0.8
4,colsample_bytree,0.8
5,objective,binary:logistic
6,eval_metric,logloss
7,random_state,42
8,missing_value_fill,median per feature
9,target_threshold,risk_delta_12m >= 8


,value
base_score,None
booster,None
callbacks,None
colsample_bylevel,None
colsample_bynode,None
colsample_bytree,0.8
device,None
early_stopping_rounds,None
enable_categorical,False
eval_metric,aucpr


,duplicated_suffix_feature


,feature,importance
0,air_heat_change_1y,0.280035
1,air_heat_change_3y,0.159205
2,hot_days_count,0.071687
3,heat_score,0.066297
4,risk_score_t,0.063339
5,temp_trend_3y,0.051068
6,year,0.041666
7,air_temp_anomaly_vs_baseline,0.040121
8,surface_heat_change_1y,0.035896
9,built_gain_1y,0.025806


,metric,value
0,top_importance_feature,air_heat_change_1y
1,mean_predicted_probability,0.061723
2,max_predicted_probability,0.878731
3,alert_rows,335
